In [1]:
# Cell 1: Imports and Advanced Setup
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score, roc_auc_score, precision_recall_curve, roc_curve
from xgboost import XGBClassifier
import shap
import pickle
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('dark_background')
sns.set_palette("husl")

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

print("All libraries imported successfully!")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")
print(f"Scikit-learn imported")
print(f"XGBoost imported")
print(f"SHAP imported")
print("Ready for Part 4 - Model Improvement!")

All libraries imported successfully!
Pandas version: 2.3.3
NumPy version: 2.4.3
Scikit-learn imported
XGBoost imported
SHAP imported
Ready for Part 4 - Model Improvement!


In [2]:
# Cell 2: Load All Datasets and Build Base Features

# Load all datasets
races = pd.read_csv('../data/races.csv')
results = pd.read_csv('../data/results.csv')
drivers = pd.read_csv('../data/drivers.csv')
constructors = pd.read_csv('../data/constructors.csv')
driver_standings = pd.read_csv('../data/driver_standings.csv')
constructor_standings = pd.read_csv('../data/constructor_standings.csv')
qualifying = pd.read_csv('../data/qualifying.csv')
pit_stops = pd.read_csv('../data/pit_stops.csv')
circuits = pd.read_csv('../data/circuits.csv')

print("All datasets loaded!")
print(f"Races: {races.shape}")
print(f"Results: {results.shape}")
print(f"Drivers: {drivers.shape}")
print(f"Qualifying: {qualifying.shape}")
print(f"Pit stops: {pit_stops.shape}")

# Filter to modern era (2020-2024)
modern_races = races[races['year'].between(2020, 2024)].copy()
modern_results = results[results['raceId'].isin(modern_races['raceId'])].copy()
modern_quali = qualifying[qualifying['raceId'].isin(modern_races['raceId'])].copy()
modern_pitstops = pit_stops[pit_stops['raceId'].isin(modern_races['raceId'])].copy()

print(f"\nFiltered to 2020-2024:")
print(f"Modern races: {modern_races.shape[0]} races")
print(f"Modern results: {modern_results.shape[0]} results")

# Merge with race info
df = modern_results.merge(modern_races[['raceId', 'year', 'round', 'circuitId', 'name']], 
                          on='raceId', how='left')

# Create target variable: is_winner (1 if finished P1, 0 otherwise)
df['is_winner'] = (df['positionOrder'] == 1).astype(int)

# Handle missing grid positions (set to last place)
df['grid'] = df['grid'].replace(0, 20)

# Base features from Part 3
df['grid_squared'] = df['grid'] ** 2
df['is_pole'] = (df['grid'] == 1).astype(int)
df['is_front_row'] = (df['grid'] <= 2).astype(int)

print(f"\nTarget variable created:")
print(f"Total winners: {df['is_winner'].sum()}")
print(f"Total non-winners: {(df['is_winner'] == 0).sum()}")
print(f"Class balance: {df['is_winner'].mean():.3f}")

print("\nBase features created: grid, grid_squared, is_pole, is_front_row")
print("Ready for advanced feature engineering!")

All datasets loaded!
Races: (1125, 18)
Results: (26759, 18)
Drivers: (861, 9)
Qualifying: (10494, 9)
Pit stops: (11371, 7)

Filtered to 2020-2024:
Modern races: 107 races
Modern results: 2139 results

Target variable created:
Total winners: 107
Total non-winners: 2032
Class balance: 0.050

Base features created: grid, grid_squared, is_pole, is_front_row
Ready for advanced feature engineering!
